# Penguins classification with auto_sklearn2 + cross-validation (no train/test split)
# Predict target: species

In [1]:
from auto_sklearn2 import AutoSklearnClassifier
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

# -----------------------------
# Config
# -----------------------------
k = 7             # number of CV folds
time_limit = 120  # seconds per AutoML fit (per fold)

# -----------------------------
# 1) Load dataset
# -----------------------------
penguins = pd.read_csv('./penguins_clean.csv')

# Target (change to your classification target)
target_col = "sex"
df = penguins.copy()

# -----------------------------
# 2) Basic cleaning: drop rows with missing values
# -----------------------------
df = df.dropna()

# -----------------------------
# 3) One-hot encode categoricals (excluding target)
# -----------------------------
feature_cols = df.drop(columns=[target_col])
categorical_features = feature_cols.select_dtypes(exclude=np.number).columns

df_enc = pd.get_dummies(df, columns=categorical_features, drop_first=True)

# Split X / y
X = df_enc.drop(columns=[target_col])
y = df_enc[target_col].reset_index(drop=True)
X = X.reset_index(drop=True)

# -----------------------------
# 4) K-fold cross-validation
# -----------------------------
kf = KFold(n_splits=k, shuffle=True)

cv_acc = []
cv_f1 = []

# Out-of-fold predictions container
oof_pred = np.full(shape=len(y), fill_value=None, dtype=object)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    automl_cv = AutoSklearnClassifier(time_limit=time_limit)
    automl_cv.fit(X_tr, y_tr)

    y_va_pred = automl_cv.predict(X_va)
    oof_pred[va_idx] = y_va_pred

    acc = accuracy_score(y_va, y_va_pred)
    f1 = f1_score(y_va, y_va_pred, average='weighted')
    cv_acc.append(acc)
    cv_f1.append(f1)
    print(f"[Fold {fold}] Accuracy={acc:.4f} | F1={f1:.4f}")

# Summary across folds
print(f"\n=== Cross-Validation Summary ({k}-fold) ===")
print(f"Accuracy Mean: {np.mean(cv_acc):.4f} | Std: {np.std(cv_acc):.4f}")
print(f"F1 Mean: {np.mean(cv_f1):.4f} | Std: {np.std(cv_f1):.4f}")

# Out-of-fold performance
oof_acc = accuracy_score(y, oof_pred)
oof_f1 = f1_score(y, oof_pred, average='weighted')
print(f"\n=== Out-of-Fold (OOF) Performance ===")
print(f"OOF Accuracy: {oof_acc:.4f} | OOF F1: {oof_f1:.4f}")
print("\nClassification Report (OOF):")
print(classification_report(y, oof_pred))

# -----------------------------
# 5) Fit final model on full dataset
# -----------------------------
auto_sklearn_full = AutoSklearnClassifier(time_limit=time_limit)
auto_sklearn_full.fit(X, y)

best_params = getattr(auto_sklearn_full, "best_params", None)
if best_params is None:
    best_params = getattr(auto_sklearn_full, "best_params_", "N/A")

print("\n=== Final Model Trained on Full Data ===")
print(f"Best params: {best_params}")

if hasattr(auto_sklearn_full, "get_models_performance"):
    print("\nModel leaderboard:")
    perf = auto_sklearn_full.get_models_performance()
    df_lb = pd.DataFrame(list(perf.items()), columns=["model", "score"])
    df_lb.sort_values("score", ascending=False, inplace=True)
    print(df_lb.head(10))



Error evaluating standard_scaler + multinomial_nb: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packa

[Fold 1] Accuracy=0.8542 | F1=0.8528


Error evaluating standard_scaler + multinomial_nb: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packa

[Fold 2] Accuracy=0.9583 | F1=0.9586


Error evaluating standard_scaler + multinomial_nb: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packa

...

[Fold 6] Accuracy=0.8723 | F1=0.8723


Error evaluating standard_scaler + multinomial_nb: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packa

[Fold 7] Accuracy=0.9149 | F1=0.9142

=== Cross-Validation Summary (7-fold) ===
Accuracy Mean: 0.9070 | Std: 0.0368
F1 Mean: 0.9068 | Std: 0.0371

=== Out-of-Fold (OOF) Performance ===
OOF Accuracy: 0.9069 | OOF F1: 0.9069

Classification Report (OOF):
              precision    recall  f1-score   support

      Female       0.91      0.90      0.91       165
        Male       0.90      0.92      0.91       168

    accuracy                           0.91       333
   macro avg       0.91      0.91      0.91       333
weighted avg       0.91      0.91      0.91       333



Error evaluating standard_scaler + multinomial_nb: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "C:\Users\biauser\anaconda3\envs\edal\Lib\site-packa


=== Final Model Trained on Full Data ===
Best params: {'preprocessor': 'robust_scaler', 'classifier': 'ada_boost'}

Model leaderboard:
                              model     score
40          robust_scaler_ada_boost  0.924966
41        robust_scaler_extra_trees  0.921981
34  robust_scaler_gradient_boosting  0.919086
24        minmax_scaler_extra_trees  0.918996
23          minmax_scaler_ada_boost  0.918951
15              standard_scaler_lda  0.918815
44         robust_scaler_linear_svc  0.916011
8       standard_scaler_extra_trees  0.915966
3               standard_scaler_svc  0.915966
5               standard_scaler_mlp  0.915920
